# Split-RAG experiments
This notebook contains the code for the final Split-RAG architecture (Experiment 3).

**Baseline RAGAs results**: 
* faithfulness: 0.7205 
* factual correctness: 0.3340
* context precision: 0.8750
* context recall: 0.3417

## Imports

In [27]:
import os
import time
import pandas as pd
import json
from datasets import Dataset
from dotenv import load_dotenv

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_openai import AzureOpenAIEmbeddings
from langchain_chroma import Chroma
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import AsyncAzureOpenAI
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_classic.storage import LocalFileStore, create_kv_docstore



from ragas.metrics import Faithfulness, FactualCorrectness, ContextPrecision, ContextRecall
from ragas.llms import llm_factory
import nest_asyncio
from ragas import aevaluate 
from ragas import RunConfig



C:\Users\verkad004\AppData\Local\Temp\ipykernel_15020\584857335.py:20: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, FactualCorrectness, ContextPrecision, ContextRecall
C:\Users\verkad004\AppData\Local\Temp\ipykernel_15020\584857335.py:20: DeprecationWarning: Importing FactualCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import FactualCorrectness
  from ragas.metrics import Faithfulness, FactualCorrectness, ContextPrecision, ContextRecall
C:\Users\verkad004\AppData\Local\Temp\ipykernel_15020\584857335.py:20: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead

## Environment variables

In [4]:
env_path = os.path.join("..", ".env")
load_dotenv(dotenv_path=env_path)

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")
embedding_deployment = os.getenv("AZURE_EMBEDDING_DEPLOYMENT")
deployment = os.getenv("AZURE_OPENAI_DEPLOYMENT")

# Initialize Azure AD token provider
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default"
)

## Data Loading
In contrast to the baseline experiments, the Split-RAG architecture employs a decoupled data strategy. Rather than treating legislation and jurisprudence as a unified corpus, this approach maintains them as two distinct, specialized knowledge bases.

In [5]:
legislation_path = '../data/legislation_omgevingswet_cleaned.jsonl'
jurisprudence_path = '../data/jurisprudence_omgevingswet_cleaned.jsonl'

common_cols = ['id', 'title', 'text', 'word_count', 'source_type']
df_legislation = pd.read_json(legislation_path, lines=True)
df_jurisprudence = pd.read_json(jurisprudence_path, lines=True)

df_leg_sub = df_legislation[common_cols]
df_jur_sub = df_jurisprudence[common_cols]

print(f"Legislation loaded: {len(df_legislation)} rows")
print(f"Jurisprudence loaded: {len(df_jurisprudence)} rows")

Legislation loaded: 725 rows
Jurisprudence loaded: 3404 rows


In [6]:
# Legislation document creation
docs_legislation = [
    Document(
        page_content=str(row['text']),
        metadata={
            "id": row['id'],
            "title": row['title'],
            "source_type": row['source_type'],
            "word_count": row['word_count']
        }
    ) for _, row in df_leg_sub.iterrows()
]

# jurisprudence document creation
docs_jurisprudence = [
    Document(
        page_content=str(row['text']),
        metadata={
            "id": row['id'],
            "title": row['title'],
            "source_type": row['source_type'],
            "word_count": row['word_count']
        }
    ) for _, row in df_jur_sub.iterrows()
]

print(f"Created {len(docs_legislation)} legislation documents.")
print(f"Created {len(docs_jurisprudence)} jurisprudence documents.")

Created 725 legislation documents.
Created 3404 jurisprudence documents.


# Retrieving documents per dataset

In [7]:
embeddings = AzureOpenAIEmbeddings(
    azure_deployment=embedding_deployment, 
    azure_endpoint=endpoint,              
    openai_api_version=api_version,       
    azure_ad_token_provider=token_provider
)

## Legislation
For legislation, we are implementing a Parent Document Retrieval (PDR) system. Instead of standard chunking, this system employs a hierarchical approach.

Laws are inherently hierarchical and logically structured. Recent research demonstrates that multi-granular context is superior to fixed chunk sizes. By indexing small child chunks (400 tokens), we maximise search precision. However, to prevent the AI from losing the legal context, the full parent article is served to the model as context upon a match. This chunk-to-context reconstruction directly addresses the shortcomings of naive RAG systems when dealing with complex legislation.


In [8]:
# Splitter for search chunks (childs)
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400, 
    chunk_overlap=50
)

# Define paths
vdb_base_path = "../data/vector_stores"
leg_child_path = os.path.join(vdb_base_path, "split_legislation_child")
leg_parent_path = os.path.join(vdb_base_path, "split_legislation_parent_store")

# Parent storage for disk
fs = LocalFileStore(leg_parent_path)
parent_docstore = create_kv_docstore(fs) 

# Child database
vdb_leg_child = Chroma(
    collection_name="leg_pdr_final",
    embedding_function=embeddings, 
    persist_directory=leg_child_path
)

# Retriever that links child chunks (cosine similarity)
retriever_leg_parent = ParentDocumentRetriever(
    vectorstore=vdb_leg_child,
    docstore=parent_docstore,
    child_splitter=child_splitter,
)

In [9]:
# Check if disk is empty and populate using unique, stable IDs
if vdb_leg_child._collection.count() == 0:
    print("Vectordatabase empty. Indexing legislation documents...")
    batch_size = 50
    
    for i in range(0, len(docs_legislation), batch_size):
        batch = docs_legislation[i : i + batch_size]
        
        # Extract fixed unique IDs from document metadata
        batch_ids = [str(doc.metadata["id"]) for doc in batch]
        
        # Populate parent docstore and child vector store simultaneously
        retriever_leg_parent.add_documents(batch, ids=batch_ids)
        
        print(f"Progress: {min(i + batch_size, len(docs_legislation))}/{len(docs_legislation)} articles done")
        time.sleep(1)
        
    # Force persistence to disk
    if hasattr(vdb_leg_child, 'persist'):
        vdb_leg_child.persist()
        
    print(f"Database is ready. Total chunks: {vdb_leg_child._collection.count()}")
else:
    print(f"Database is ready. Loaded {vdb_leg_child._collection.count()} chunks from disk.")

Database is ready. Loaded 4100 chunks from disk.


## Jurisprudence
For the jurisprudence, we use Structure-Aware Chunking.

Jurisprudence is narrative and story-driven in nature, meaning that relevant information is often scattered across lengthy texts. Standard retrieval is often insufficient for this type of document. A splitter is used that preserves the logical integrity of the judgment (splitting by ECLI and paragraphs). 

In [ ]:
structure_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=150,
    length_function=len,
    separators=[
        "\nECLI:",          
        "\n\n",            
        "\n",             
        ". ",              
        " "               
    ]
)

# Apply splitter to jurisprudence documents
chunks_juris_structure = structure_splitter.split_documents(docs_jurisprudence)

print(f"Jurisprudence: Structure-Aware Results")
print(f"Totaal aantal chunks: {len(chunks_juris_structure)}")

# Check first chunk
if chunks_juris_structure:
    print(f"\nSample Metadata: {chunks_juris_structure[0].metadata}")
    print(f"Sample Content: {chunks_juris_structure[0].page_content[:300]}...")

Jurisprudence: Structure-Aware Results
Totaal aantal chunks: 51321

Sample Metadata: {'id': 'ECLI:NL:RBGEL:2024:26', 'title': 'ECLI:NL:RBGEL:2024:26, Rechtbank Gelderland, 05-01-2024, AWB-22_5249 en 22_5252', 'source_type': 'jurisprudence', 'word_count': 2403}
Sample Content: Weigering handhavingsverzoeken m.b.t. geitenhouderij. Intern salderen. Beroep gegrond vanwege een motiveringsgebrek. De rechtsgevolgen worden door de rechtbank in stand gelaten omdat in het verweerschrift afdoende is onderbouwd dat er geen sprake is van een overtreding van de Wnb.


    

Zittingspl...


In [ ]:
def create_vdb_in_batches(chunks, path, name, embeddings, batch_size=50):
    print(f"Checking existing progress for: {name}")
    
    # Set up the database connection
    vector_db = Chroma(
        persist_directory=path,
        embedding_function=embeddings
    )
    
    existing_count = vector_db._collection.count()
    
    # See if we can resume or need to start fresh
    if existing_count > 0:
        print(f"Resuming: Found {existing_count} chunks already in database.")
        remaining_chunks = chunks[existing_count:]
    else:
        print("No existing data found. Starting from scratch.")
        remaining_chunks = chunks

    if not remaining_chunks:
        print(f"All chunks for {name} are already processed.\n")
        return vector_db

    print(f"Adding {len(remaining_chunks)} remaining chunks...")
    
    # Loop through the remaining chunks in batches
    for i in range(0, len(remaining_chunks), batch_size):
        batch = remaining_chunks[i : i + batch_size]
        vector_db.add_documents(documents=batch)
        
        current_total = existing_count + i + len(batch)
        print(f"Progress for {name}: {current_total}/{len(chunks)} chunks total.")
        
        # Pause to avoid API rate limits and let disk save data
        time.sleep(1) 
        
    print(f"{name} successfully updated at {path}\n")
    return vector_db

# Jurisprudence configuration
juris_vdb_path = os.path.join(vdb_base_path, "split_jurisprudence_structure")

vdb_juris_split = create_vdb_in_batches(
    chunks=chunks_juris_structure, 
    path=juris_vdb_path, 
    name="Jurisprudence-Structure",
    embeddings=embeddings,
    batch_size=50
)

Checking existing progress for: Jurisprudence-Structure
Hervatten: Er staan al 51321 chunks in de database.
Alle chunks voor Jurisprudence-Structure zijn al verwerkt!



In [12]:
# Load database
vdb_juris_split = Chroma(
    persist_directory=os.path.join(vdb_base_path, "split_jurisprudence_structure"),
    embedding_function=embeddings
)

print(f"Jurisprudence chunks geladen: {vdb_juris_split._collection.count()}")

Jurisprudence chunks geladen: 51321


In [13]:
retriever_juris_structure = vdb_juris_split.as_retriever(
    search_type="mmr", 
    search_kwargs={
        "k": 15,                # fetch 15 fragments
        "fetch_k": 50,          # fetch 50 and retrieve 15 most diverse
        "lambda_mult": 0.5      # balance between relevance and diversity
    }
)

## Split-RAG

In [14]:
vdb_path = "../data/vector_stores"

vdb_leg_child = Chroma(
    persist_directory=os.path.join(vdb_path, "split_legislation_child"),
    embedding_function=embeddings
)

# Load parent store (complete articles)
fs_leg = LocalFileStore(os.path.join(vdb_path, "split_legislation_parent_store"))
parent_docstore = create_kv_docstore(fs_leg)

vdb_juris_split = Chroma(
    persist_directory=os.path.join(vdb_path, "split_jurisprudence_structure"),
    embedding_function=embeddings
)

# Check chunk counts for your Split-RAG architecture
print(f"Split-RAG Legislation chunks: {vdb_leg_child._collection.count()}")
print(f"Split-RAG Jurisprudence chunks: {vdb_juris_split._collection.count()}")

Split-RAG Legislation chunks: 2050
Split-RAG Jurisprudence chunks: 51321


## RAGAs evaluation

### QA pairs

In [15]:
# Load JSON file
file_path = "../data/QA_pairs_evaluation.json" 

with open(file_path, 'r', encoding='utf-8') as f:
    qa_list = json.load(f)

# Convert to DataFrame
df_qa = pd.DataFrame(qa_list)
print(f"Dataset geladen: {len(df_qa)} vragen gevonden.")

Dataset geladen: 10 vragen gevonden.


### RAGAs dataset generations

In [16]:
client = AsyncAzureOpenAI(
    azure_endpoint=endpoint,
    azure_deployment=deployment,
    api_version=api_version,
    azure_ad_token_provider=token_provider,
)

In [17]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[retriever_leg_parent, retriever_juris_structure],
    weights=[0.6, 0.4] 
)

In [18]:
async def run_evaluation_loop(retriever, dataset_list, strategy_name): 
    questions = []
    all_contexts = []
    ground_truths = []
    all_answers = []

    system_prompt = (
    "Je bent een juridisch assistent voor de Gemeente Amsterdam.\n"
    "Beantwoord de vraag uitsluitend op basis van de verstrekte context.\n\n"
    "EISEN:\n"
    "1. Noem het specifieke wetsartikel uit de Omgevingswet.\n"
    "2. Noem het ECLI-nummer van de relevante uitspraak.\n"
    "3. Als informatie ontbreekt, geef dit dan expliciet aan."
)


    print(f"Start retrieval and generation for: {strategy_name}...")

    for item in dataset_list:
        q = item['question']
        gt = item['ground_truth']
        
        # Retrieval
        docs = retriever.invoke(q) 
        ctx_list = [doc.page_content for doc in docs]
        context_text = "\n\n".join(ctx_list)
        
        # Generation
        resp = await client.chat.completions.create(
            model=deployment,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Context: {context_text}\n\nVraag: {q}"},
            ],
            temperature=0 
        )
        answer = resp.choices[0].message.content
        
        # Save to lists
        questions.append(q)
        all_contexts.append(ctx_list) 
        ground_truths.append(gt)
        all_answers.append(answer)

    # Dataset object for RAGAs
    ds = Dataset.from_dict({
        "question": questions,
        "answer": all_answers,
        "contexts": all_contexts,
        "ground_truth": ground_truths
    })
    
    os.makedirs("../data/results", exist_ok=True)
    ds.to_pandas().to_csv(f"../data/results/results_{strategy_name}.csv", index=False, encoding='utf-16')
    
    return ds

## Results and analysis

In [19]:
evaluator_llm = llm_factory(
    model=deployment,
    client=client,
    max_tokens=4096
    )

config = RunConfig(
    timeout=240,     
    max_retries=20, 
    max_wait=60,
    max_workers=1,
    seed=42,      
)

nest_asyncio.apply()

# Initialize metrics
metrics = [
    Faithfulness(llm=evaluator_llm),
    FactualCorrectness(llm=evaluator_llm), 
    ContextPrecision(llm=evaluator_llm),
    ContextRecall(llm=evaluator_llm)
]




In [25]:
dataset_ensemble = await run_evaluation_loop(ensemble_retriever, qa_list, "hybrid-split-rag-winner")

result_ensemble = await aevaluate(
    dataset=dataset_ensemble,
    metrics=metrics,
    run_config=config
)

print(f"results split RAG ensemble: {result_ensemble}")

Start retrieval and generation for: hybrid-split-rag-winner...


C:\Users\verkad004\AppData\Local\Temp\ipykernel_15020\2054068488.py:3: DeprecationWarning: aevaluate() is deprecated and will be removed in a future version. Use the @experiment decorator instead. See https://docs.ragas.io/en/latest/concepts/experiment/ for more information.
  result_ensemble = await aevaluate(
Evaluating: 100%|██████████| 40/40 [12:41<00:00, 19.04s/it]

results split RAG ensemble: {'faithfulness': 0.8724, 'factual_correctness(mode=f1)': 0.3400, 'context_precision': 0.5051, 'context_recall': 0.7150}


In [26]:
# Load split-rag data
df_split = pd.read_csv('../data/results/results_hybrid-split-rag-winner.csv',encoding='utf-16')


# Print split-rag data
for i in range(len(df_split)):
    print(f"Case {i+1}\n")
    print(f"Question:\n{df_split.loc[i, 'question']}\n")
    
    print(f"Answer:\n")
    print(f"{df_split.loc[i, 'answer']}")
    print("-" * 50)
    

Case 1

Question:
Kan een omgevingsvergunning voor een dakterras op een gemeentelijk monument worden verleend als het hekwerk de maximale bouwhoogte overschrijdt?

Answer:

Op basis van de verstrekte context kan een omgevingsvergunning voor een dakterras op een gemeentelijk monument niet worden verleend als het hekwerk de maximale bouwhoogte overschrijdt, tenzij er sprake is van een afwijking die voldoet aan de relevante planregels en het afwijkingenbeleid. 

### Toelichting:
1. **Artikel 16.15a Omgevingswet**: Dit artikel regelt de verplichte aanwijzing van adviseurs bij omgevingsvergunningen, maar biedt geen directe grondslag voor het toestaan van een dakterras dat de maximale bouwhoogte overschrijdt.

2. **Artikel 16.58 Omgevingswet**: Dit artikel stelt dat bij een gemeentelijk monument overleg met de eigenaar vereist is en dat bij wezenlijke belangen van godsdienst of levensovertuiging instemming van de eigenaar nodig is. Dit artikel is echter niet direct relevant voor de vraag ove